# Notebook 1: Evacuation Distance Analysis

This notebook computes origin-destination distances for fire evacuees using GPS mobility data and Colorado census block group (CBG) shapefiles.

**Paper reference:** Figure 1c, 1d — Spatial distribution of evacuation destinations and histogram of evacuation distances.

**Inputs:** `evacuees.csv`, `tl_2019_08_bg/` (TIGER shapefile)

**Outputs:** Evacuation distance per individual (`distance_OD` column)


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Point

In [ ]:
evacuees = pd.read_csv("evacuees.csv")

In [ ]:
shapefile_path = 'tl_2019_08_bg'
cbg_geo = gpd.read_file(shapefile_path)

cbg_geo['GEOID'] = cbg_geo['GEOID'].str.lstrip('0')

columns_keep = ["GEOID", "geometry"]
cbg_geo = cbg_geo[columns_keep]

cbg_geo.head()

In [ ]:
evacuees['pre_crisis_home_cbg'] = evacuees['pre_crisis_home_cbg'].astype(str).str.lstrip('0')
evacuees['crisis_home_cbg'] = evacuees['crisis_home_cbg'].astype(str).str.lstrip('0')

In [ ]:
# Merge the origin geometries with the evacuees DataFrame
evacuees = pd.merge(evacuees, cbg_geo, 
                    left_on='pre_crisis_home_cbg', right_on='GEOID', 
                    how='left').rename(columns={'geometry': 'origin_geometry'})

# Merge the destination geometries with the evacuees DataFrame
evacuees = pd.merge(evacuees, cbg_geo, 
                    left_on='crisis_home_cbg', right_on='GEOID', 
                    how='left').rename(columns={'geometry': 'destination_geometry'})

# Drop the duplicate 'GEOID' columns from the merges
evacuees = evacuees.drop(columns=['GEOID_x', 'GEOID_y'])

evacuees.head()

In [ ]:
evacuees = gpd.GeoDataFrame(evacuees, geometry='origin_geometry')
evacuees = gpd.GeoDataFrame(evacuees, geometry='destination_geometry')

evacuees['origin_geometry'] = evacuees['origin_geometry'].to_crs(epsg=3857)
evacuees['destination_geometry'] = evacuees['destination_geometry'].to_crs(epsg=3857)

In [ ]:
# Calculate centroids for origin and destination geometries
evacuees['origin_centroid'] = evacuees['origin_geometry'].centroid
evacuees['destination_centroid'] = evacuees['destination_geometry'].centroid


In [ ]:
from shapely.geometry import Point

def calculate_distance(row):
    if isinstance(row['origin_centroid'], Point) and isinstance(row['destination_centroid'], Point):
        return row['origin_centroid'].distance(row['destination_centroid'])
    else:
        return None

# Calculate distances and add a new column for distances
evacuees['distance_OD'] = evacuees.apply(calculate_distance, axis=1)

evacuees.head()

In [ ]:
geo_path = 'Shapefile_census/perimeter.geojson'
perimeter= gpd.read_file(geo_path)
perimeter

In [ ]:
perimeter_gdf = gpd.GeoDataFrame(perimeter, geometry='geometry')

In [ ]:
perimeter_gdf['geometry'] = perimeter_gdf['geometry'].to_crs(epsg=3857)

In [ ]:
perimeter_centroid = perimeter_gdf.geometry.centroid.iloc[0]

In [ ]:
evacuees['dist_epicentre'] = evacuees["origin_centroid"].geometry.centroid.distance(perimeter_centroid)

In [ ]:
evacuees['Or_area_sq_km'] = evacuees['origin_geometry'].area / 10**6
# Calculate population density (people per square kilometer) - or
evacuees['Or_population_density'] = evacuees['Or_total_population'] / evacuees['Or_area_sq_km']

evacuees['D1_area_sq_km'] = evacuees['destination_geometry'].area / 10**6
# Calculate population density (people per square kilometer) - dest
evacuees['D1_population_density'] = evacuees['D1_total_population'] / evacuees['D1_area_sq_km']


In [ ]:
evacuees['dist_epicentre'].hist(bins=50)
plt.title('Histogram of col1')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
bin_edges = range(0, int(evacuees['dist_epicentre'].max()) + 10000, 10000)
bin_labels = [f"{i}-{i+10000}" for i in bin_edges[:-1]]

evacuees['distance_bin'] = pd.cut(evacuees['dist_epicentre'], bins=bin_edges, labels=bin_labels, right=False)

bin_counts = evacuees['distance_bin'].value_counts(normalize=True).sort_index() * 100

bin_counts.plot(kind='bar')
plt.title('Percentage of Evacuees by Distance from Epicenter')
plt.xlabel('Distance Range (meters)')
plt.ylabel('Percentage of Evacuees')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
within_20k = evacuees[evacuees['dist_epicentre'] < 20000]

In [ ]:
percent_within_20k = (len(within_20k) / len(evacuees)) * 100

In [ ]:
percent_within_20k

In [ ]:
bin_counts

In [ ]:
evacuees.to_csv("evacuess_for_homophily_final.csv")